# Part 0 — 환경 점검

**목적** 학습을 시작하기 전에 이 컴퓨터에서 무엇이 가능한지 숫자로 확인한다.

**왜 필요한가** 배치 크기, 입력 길이, 정밀도(bf16/fp32) 선택이 전부 GPU 메모리와 연산 지원 여부에 달려 있다.
이걸 모르고 설정을 잡으면 OOM이나 NaN으로 시간을 버린다.

**실행한 것** `python scripts/check_env.py` → 결과는 `artifacts/env_report.json`.
이 노트북은 그 결과 파일을 읽어서 보여주기만 한다 (로직은 스크립트에 있다).

In [1]:
import json
from pathlib import Path
import pandas as pd

report = json.loads(Path("../artifacts/env_report.json").read_text(encoding="utf-8"))
print("생성 시각:", report["created"])

생성 시각: 2026-09-19 13:23:06


## 1. 하드웨어와 라이브러리

확인할 것: GPU가 잡히는가, VRAM은 얼마인가, **bf16을 지원하는가**.
T5 계열은 fp16에서 NaN이 잘 나기 때문에 bf16 지원 여부가 정밀도 선택을 결정한다.

In [2]:
pd.Series({"python": report["python"], "torch": report["torch"], "precision": report["precision"],
           **{f"gpu.{k}": v for k, v in report["gpu"].items()}}).to_frame("값")

,값
python,3.11.16
torch,2.10.0+cu128
precision,bf16-autocast
gpu.available,True
gpu.name,NVIDIA GeForce RTX 5060 Laptop GPU
gpu.vram_total_gb,7.93
gpu.compute_capability,12.0
gpu.bf16_supported,True
gpu.cuda_version,12.8
gpu.cudnn_version,91002


In [3]:
# None 이면 미설치. kiwipiepy 는 형태소 기준 ROUGE 용 선택 패키지 (Part 5 에서 필요)
pd.Series(report["libs"]).to_frame("버전")

,버전
transformers,4.57.1
sentencepiece,0.2.2
rouge_score,installed
ijson,3.5.1
yaml,6.0.3
datasets,4.8.5
accelerate,1.15.0
kiwipiepy,None


## 2. 토크나이저 첫 인상

`paust/pko-t5-base` 토크나이저로 짧은 대화 한 줄을 쪼개 본다. 자세한 분석은 Part 3에서 한다.

관찰할 점
- UNK가 0개인가 (바이트 BPE라 원리상 UNK가 거의 없다)
- `ㅋㅋ`가 어떻게 쪼개지는가 → 호환 자모 `ㅋ`가 초성 자모 `ᄏ`로 정규화되고 **글자 하나당 토큰 하나**가 된다.
  `ㅋㅋㅋㅋㅋㅋ`는 그대로 6토큰이므로, 반복 자모 축약(`collapse_repeat_jamo`)이 길이 절약에 효과가 있다
- `#@이름#` 비식별화 태그는 `#`, `@`, `이름`, `#` 4토큰으로 쪼개진다

In [4]:
tc = report["tokenizer_check"]
print("원문:", repr(tc["text"]))
print("토큰 수:", tc["n_tokens"], "/ UNK:", tc["unk"])
print(" | ".join(repr(p) for p in tc["pieces"]))

원문: 'P01: 오늘 저녁 뭐 먹을래? ㅋㅋ\nP02: 치킨 어때 #@이름#'
토큰 수: 31 / UNK: 0
'P' | '0' | '1' | ':' | ' ' | '오늘' | ' ' | '저녁' | ' ' | '뭐' | ' ' | '먹을래' | '?' | ' ' | 'ᄏ' | 'ᄏ' | '\n' | 'P' | '0' | '2' | ':' | ' ' | '치킨' | ' ' | '어때' | ' ' | '#' | '@' | '이름' | '#' | '</s>'


## 3. 더미 배치 학습 — 메모리 설정 비교

사전학습 모델(2.76억 파라미터)을 올리고, **최대 길이로 꽉 채운 가짜 배치**로 학습 3스텝을 돌렸다.
정밀도는 Trainer의 `bf16=True`와 같은 방식(가중치 fp32 + bf16 autocast)이다.

- 입력을 꽉 채웠으므로 이 값은 **최악의 경우 메모리**다. 실제 대화는 짧아서(글자 수 p95 242) 동적 패딩을 쓰면 훨씬 덜 쓴다
- 1스텝째에는 AdamW 상태(파라미터당 2개)가 아직 없어서 메모리가 적다. 그래서 3스텝을 잰다
- `fits_vram`: peak가 전체 VRAM의 95% 미만인가. Windows 드라이버는 VRAM이 모자라면 에러 대신 **시스템 RAM으로 넘겨서** 조용히 느려진다

In [5]:
df = pd.DataFrame(report["smoke"])
df["last_step_sec"] = df["step_sec"].str[-1]
df["sec_per_sample"] = (df["last_step_sec"] / df["batch_size"]).round(4)
df[["batch_size", "max_source_length", "optimizer", "gradient_checkpointing",
    "peak_vram_gb", "fits_vram", "step_sec", "sec_per_sample", "losses"]]

,batch_size,max_source_length,optimizer,gradient_checkpointing,peak_vram_gb,fits_vram,step_sec,sec_per_sample,losses
0,8,384,adamw,False,8.54,False,"[0.995, 3.952, 5.255]",0.6569,"[15.1606, 12.4934, 12.0004]"
1,4,384,adamw,False,6.12,True,"[0.32, 0.248, 0.235]",0.0588,"[15.5662, 12.5035, 12.1968]"
2,8,256,adamw,False,6.94,True,"[0.303, 0.3, 0.263]",0.0329,"[15.7892, 12.5861, 12.0869]"
3,8,384,adamw,True,5.21,True,"[0.481, 0.427, 0.455]",0.0569,"[15.4298, 12.7092, 12.1346]"
4,8,384,adafactor,False,6.46,True,"[0.48, 0.358, 0.371]",0.0464,"[14.897, 12.6501, 11.882]"


### 읽는 법

- **batch 8 × 입력 384 + AdamW (base.yaml 원래 기본값)**: peak 8.54GB > VRAM 7.93GB.
  스텝 시간이 1.0초 → 4.0초 → 5.3초로 늘어난다. 옵티마이저 상태가 생긴 2스텝째부터 시스템 RAM으로 넘친 것이다.
  에러가 나지 않으므로 **모르고 쓰면 학습이 20배 느려진다.**
- 나머지 4개 조합은 모두 VRAM 안에 들어간다. 샘플당 시간은 batch 8 × 256이 가장 빠르고,
  gradient checkpointing은 메모리를 가장 적게 쓰는 대신 재계산 때문에 느리다.
- loss는 모든 조합에서 유한값이고 3스텝 동안 내려간다 → forward/backward/optimizer가 정상 동작한다.
  (입력이 무작위 토큰이므로 loss 값 자체에는 의미가 없다)

### 결정

| 항목 | 내용 |
|---|---|
| 선택 | `per_device_train_batch_size` 8→**4**, `gradient_accumulation_steps` 2→**4** (유효 배치 16 유지), AdamW + bf16 유지 |
| 대안 | batch 8 × 입력 256 (가장 빠름) / gradient checkpointing / Adafactor |
| 근거 | batch 4 × 384 peak 6.12GB (여유 1.8GB). 입력 길이를 아직 정하지 않았으므로 길이를 줄이는 대안은 보류 |
| 되돌릴 조건 | Part 1에서 토큰 길이 p95가 256 이하로 나오면 `max_source_length`를 줄이고 batch 8로 되돌린다 |